# `commodities(base, start, end=None, quote="USD")`

Daily prices for 28 physical commodities, from FMP via `commodities.prices`, in USD or any currency `forex()` can price. Change the arguments and re-run any cell.

> **About the data shown.** This function's data comes from FMP's API, and FMP's license doesn't allow displaying it publicly. So the outputs saved here are synthetic: made-up data from [`_synthetic.py`](_synthetic.py), run through the function's own, unchanged code. The values are not real prices or yields.
>
> To see real data, start Jupyter with `ARORA_REAL_DATA=1`: every cell then reads prod, read-only, as `readonly_user`. Don't commit those outputs; `tests/test_notebooks_offline.py` fails if you save them.

In [1]:
# Synthetic data in place of FMP's (see the note above), unless Jupyter was started with ARORA_REAL_DATA=1.
import _synthetic

_synthetic.use()

Data: synthetic


In [2]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.commodities import ALIASES, COMMODITIES, CommodityBar
from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import commodities


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The 28 commodities
Call them by root code, name or alias. Each price is for one of `unit`, from the CME, ICE and LME contract specs. The table's financial futures (ES, NQ, YM, RTY, ZB, ZN, ZF, ZT, ZQ, DX) and the micro gold and silver contracts (MGC, SIL, which mirror GC and SI) are not served.

In [3]:
pl.DataFrame([{"code": c, "name": n, "symbol": s, "unit": u} for c, (s, u, n) in COMMODITIES.items()])

code,name,symbol,unit
str,str,str,str
"""BZ""","""brent crude""","""BZUSD""","""barrel"""
"""CL""","""wti crude""","""CLUSD""","""barrel"""
"""HO""","""heating oil""","""HOUSD""","""gallon"""
"""RB""","""rbob gasoline""","""RBUSD""","""gallon"""
"""NG""","""natural gas""","""NGUSD""","""MMBtu"""
…,…,…,…
"""LB""","""lumber""","""LBUSD""","""1,000 board feet"""
"""LE""","""live cattle""","""LEUSX""","""lb"""
"""GF""","""feeder cattle""","""GFUSX""","""lb"""


## By code, name or alias
A name or alias works anywhere a code does, in any case and spacing. Every bar carries the code and the primary name. Words that could mean two commodities (`oil`, `gas`, `cattle`) are not aliases.

In [4]:
week = today_et() - timedelta(days=7)
assert commodities("GC", week) == commodities("gold", week) == commodities("  Gold ", week)
show(commodities("crude", week)[-2:] + commodities("Wheat", week)[-2:])

4 rows

commodity|name|quote|date|route|quote_date|open|high|low|close|volume|unit
---|---|---|---|---|---|---|---|---|---|---|---
CL|wti crude|USD|2026-11-20|stored|None|111.514813|113.109418|110.327629|111.339565|6551|USD per barrel
CL|wti crude|USD|2026-11-23|stored|None|111.288057|113.168438|110.088652|113.080042|2908|USD per barrel
KE|hard red winter wheat|USD|2026-11-20|stored|None|0.85125445|0.86317718|0.84835029|0.85905468|7601|USD per bushel
KE|hard red winter wheat|USD|2026-11-23|stored|None|0.85640357|0.86372358|0.83891489|0.84492523|2998|USD per bushel

In [5]:
pl.DataFrame({"alias": list(ALIASES), "code": list(ALIASES.values())})

alias,code
str,str
"""crude""","""CL"""
"""crude oil""","""CL"""
"""wti""","""CL"""
"""brent""","""BZ"""
"""brent crude oil""","""BZ"""
…,…
"""sugar 11""","""SB"""
"""hogs""","""HE"""
"""lean hog""","""HE"""


## In USD, the default
FMP daily bars as stored (`route` = stored): open, high, low, close and volume. `quote_date` is blank because no rate is used.

In [6]:
show(commodities("GC", today_et() - timedelta(days=7)))

6 rows

commodity|name|quote|date|route|quote_date|open|high|low|close|volume|unit
---|---|---|---|---|---|---|---|---|---|---|---
GC|gold|USD|2026-11-16|stored|None|106.660345|107.234435|106.518665|106.983389|4566|USD per troy oz
GC|gold|USD|2026-11-17|stored|None|107.530521|108.523194|107.096828|108.196496|1552|USD per troy oz
GC|gold|USD|2026-11-18|stored|None|108.589885|110.636367|108.512806|109.663691|1536|USD per troy oz
GC|gold|USD|2026-11-19|stored|None|109.861420|110.384084|109.409625|109.497134|9290|USD per troy oz
GC|gold|USD|2026-11-20|stored|None|109.474742|110.088538|108.735563|109.404688|5360|USD per troy oz
GC|gold|USD|2026-11-23|stored|None|109.577879|110.477207|109.234074|109.538498|6518|USD per troy oz

## A symbol FMP quotes in cents
Corn is stored as ZCUSX, in US cents per bushel. It comes back in dollars.

In [7]:
show(commodities("ZC", today_et() - timedelta(days=7)))

6 rows

commodity|name|quote|date|route|quote_date|open|high|low|close|volume|unit
---|---|---|---|---|---|---|---|---|---|---|---
ZC|corn|USD|2026-11-16|stored|None|0.88181397|0.88666121|0.87755058|0.88516124|8669|USD per bushel
ZC|corn|USD|2026-11-17|stored|None|0.88462335|0.89337786|0.88199394|0.88874968|7362|USD per bushel
ZC|corn|USD|2026-11-18|stored|None|0.89145352|0.90138335|0.88999999|0.89246604|6099|USD per bushel
ZC|corn|USD|2026-11-19|stored|None|0.88704492|0.89200677|0.8688369|0.87660858|8503|USD per bushel
ZC|corn|USD|2026-11-20|stored|None|0.87859329|0.89914455|0.87353321|0.8929919|3103|USD per bushel
ZC|corn|USD|2026-11-23|stored|None|0.8902865|0.90102621|0.88942725|0.89516748|3633|USD per bushel

## In another currency
The USD price times `forex("USD", "EUR")` that day (`route` = crossed). Open uses the day's opening rate, close its closing rate. High and low are blank: the commodity and the rate peak at different moments. Volume is contracts of the commodity, so it stays.

In [8]:
show(commodities("CL", today_et() - timedelta(days=7), quote="EUR"))

6 rows

commodity|name|quote|date|route|quote_date|open|high|low|close|volume|unit
---|---|---|---|---|---|---|---|---|---|---|---
CL|wti crude|EUR|2026-11-16|crossed|2026-11-16|99.2931820289|None|None|98.6463642115|2903|EUR per barrel
CL|wti crude|EUR|2026-11-17|crossed|2026-11-17|98.8682889626|None|None|97.8936694503|7019|EUR per barrel
CL|wti crude|EUR|2026-11-18|crossed|2026-11-18|97.0816914097|None|None|99.6312810260|1260|EUR per barrel
CL|wti crude|EUR|2026-11-19|crossed|2026-11-19|100.113980308|None|None|100.203806285|5160|EUR per barrel
CL|wti crude|EUR|2026-11-20|crossed|2026-11-20|99.9883553697|None|None|98.3517276104|6551|EUR per barrel
CL|wti crude|EUR|2026-11-23|crossed|2026-11-23|98.5626351728|None|None|99.1574443905|2908|EUR per barrel

## Either way round a pair is stored
EUR is stored as EURUSD and JPY as USDJPY; `forex()` handles both, so any currency it prices works.

In [9]:
start = today_et() - timedelta(days=7)
rows = [r for q in ["USD", "EUR", "JPY", "GBP", "INR", "CHF"] for r in commodities("GC", start, quote=q)]
to_df(rows, CommodityBar).pivot(on="unit", index="date", values="close")

date,USD per troy oz,EUR per troy oz,JPY per troy oz,GBP per troy oz,INR per troy oz,CHF per troy oz
date,"decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]"
2026-11-16,106.9833890000,96.6862982377,94.0228863396,95.8672892138,97.9967843240,114.2606130870
2026-11-17,108.1964960000,97.5535940016,95.3969587197,96.7669629394,98.6437191717,115.1400061310
2026-11-18,109.6636910000,98.4070996816,96.1500536855,98.2465568724,100.5946134180,116.9824264100
2026-11-19,109.4971340000,98.1622417795,95.9712815284,98.1244160549,100.7651755520,116.2308792500
2026-11-20,109.4046880000,96.6425553528,95.1403953739,97.2650385398,100.4061524120,115.4398882090
2026-11-23,109.5384980000,96.0519410141,94.9406309870,97.8412795106,99.9017391000,116.4775427710


## Days with no rate carry the last one
A commodity can trade on a day its currency pair has no bar. That day uses the pair's last close, for open and close alike, and `quote_date` names the day the rate came from. Here: every carried day for gold in EUR over the last five years.

In [10]:
gc_eur = to_df(commodities("GC", today_et() - timedelta(days=5 * 365), quote="EUR"), CommodityBar)
carried = gc_eur.filter(pl.col("quote_date") != pl.col("date"))
print(f"{carried.height} of {gc_eur.height} days carried a rate")
carried.select("date", "quote_date", "open", "close").tail(10)

6 of 1304 days carried a rate


date,quote_date,open,close
date,date,"decimal[38,10]","decimal[38,10]"
2023-12-25,2023-12-22,103.1531824990,103.1234927440
2024-01-01,2023-12-29,107.1794576910,106.5660401490
2024-12-25,2024-12-24,87.8791838096,88.6342091035
2025-01-01,2024-12-31,90.9387557594,90.5174510787
2025-12-25,2025-12-24,115.2553220920,117.1483924650
2026-01-01,2025-12-31,120.1468414930,121.6282828160


## The first days look back before start
Starting on a carried day still gives that day a price: the rate is looked up as far as 14 days before `start`.

In [11]:
first = carried["date"][-1]
show(commodities("GC", first, first + timedelta(days=3), quote="EUR"))

2 rows

commodity|name|quote|date|route|quote_date|open|high|low|close|volume|unit
---|---|---|---|---|---|---|---|---|---|---|---
GC|gold|EUR|2026-01-01|crossed|2025-12-31|120.146841493|None|None|121.628282816|8641|EUR per troy oz
GC|gold|EUR|2026-01-02|crossed|2026-01-02|122.196999436|None|None|122.603646756|3601|EUR per troy oz

## Before the currency's history, days are skipped
FMP's forex history starts on 1999-01-01, so gold in EUR starts there while gold in USD goes back further. Nothing is invented for the earlier days.

In [12]:
span = date(1998, 12, 24), date(1999, 1, 8)
print("USD:", [r.date.isoformat() for r in commodities("GC", *span)])
print("EUR:", [r.date.isoformat() for r in commodities("GC", *span, quote="EUR")])

USD: ['1998-12-24', '1998-12-25', '1998-12-28', '1998-12-29', '1998-12-30', '1998-12-31', '1999-01-01', '1999-01-04', '1999-01-05', '1999-01-06', '1999-01-07', '1999-01-08']
EUR: ['1999-01-04', '1999-01-05', '1999-01-06', '1999-01-07', '1999-01-08']


## `commodities("ALL", start, end, quote)`
All 28 in one currency, ordered by code then date. Each symbol is read once, and the currency rate once.

In [13]:
to_df(commodities("ALL", today_et() - timedelta(days=7), quote="JPY"), CommodityBar).pivot(
    on="commodity", index="date", values="close")

date,ALI,BZ,CC,CL,CT,DC,GC,GF,HE,HG,HO,KC,KE,LB,LE,NG,OJ,PA,PL,RB,SB,SI,ZC,ZL,ZM,ZO,ZR,ZS
date,"decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]"
2026-11-16,102.356508474000,88.578476288600,99.033615723400,95.928958488700,0.872145196943,88.625584674400,94.022886339600,0.811386101408,1.020951088740,87.987217798800,113.037470580000,1.022863354180,0.764032313016,82.025518968000,0.767142967558,93.072457388400,0.860041913482,103.944170578000,128.838873933000,73.810474586800,1.085172705990,96.534914070700,0.777928381580,0.806544322231,99.189993909000,1.157153233090,92.232111177900,0.616146591197
2026-11-17,100.540387073000,89.506614998700,99.421298658700,95.729516057800,0.881013105697,89.707405649200,95.396958719700,0.810748369180,1.020875783530,88.651918737000,112.259015990000,1.025175336740,0.767220492492,80.986122808400,0.774904957558,93.366865091400,0.869338299965,105.918228862000,130.975138810000,76.083837915700,1.071752267740,98.030266323700,0.783611481606,0.801600738939,100.967309342000,1.148587977950,92.527834011300,0.616947809009
2026-11-18,98.819868264100,89.348211698100,98.380757173600,97.346157445900,0.870046369665,90.672775708100,96.150053685500,0.798108954306,1.017977314240,85.508766612800,110.717933473000,1.026659584040,0.748446080658,80.487880902100,0.776833862533,92.595266658300,0.870472060007,106.091556584000,129.764100728000,77.548033699000,1.062354927360,97.087463237000,0.782489234823,0.799965133236,99.334591295000,1.135335008750,91.934934594300,0.611425555333
2026-11-19,98.736392572400,88.684078158800,98.466471317900,97.967278750100,0.872035697672,89.795736035300,95.971281528400,0.782970395829,1.009801976350,84.801661246200,109.990168286000,1.026083216090,0.747745813668,79.898261094800,0.782750313459,94.660975428700,0.872869346202,104.324889597000,132.656339415000,78.270599898400,1.074605856960,97.172945293800,0.768323751938,0.792446611552,99.333779916200,1.134980612740,92.482766482100,0.604411924319
2026-11-20,98.621154993400,89.348976327200,97.767232616300,96.823001175700,0.858986942448,89.361542321700,95.140395373900,0.772772897396,0.982800165040,82.565990738500,109.437670910000,1.016102850770,0.747050271767,79.976214918600,0.791222794973,95.708179184800,0.890578661492,104.785434491000,131.102893430000,76.845608259100,1.081985386220,96.120049005200,0.776562723086,0.784110859475,98.757647782700,1.121106101340,90.117794310400,0.605952136691
2026-11-23,97.968467381900,87.809791512000,96.397624761200,98.010204042800,0.868611427094,88.584435864200,94.940630987000,0.774308233159,0.978166382954,82.985195939800,110.257355416000,1.013252896220,0.732324579374,80.144546137500,0.782356091749,93.195012523600,0.901513495842,105.007030995000,130.455727250000,76.494628599800,1.092660408150,93.578220318200,0.775871195443,0.769611233685,100.267480131000,1.115824627630,90.941308241800,0.605584726309


## Full history in one call
Gold in USD from the start of FMP's history. There is no row limit.

In [14]:
gold = to_df(commodities("GC", date(1970, 1, 1)), CommodityBar)
print(f"{gold.height:,} days, {gold['date'].min()} to {gold['date'].max()}")
(
    gold.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True)
    .agg(pl.col("date").last().alias("last_day"), pl.col("close").last())
    .tail(10)
)

14,842 days, 1970-01-02 to 2026-11-23


year,last_day,close
i32,date,"decimal[38,6]"
2017,2017-12-29,115.739792
2018,2018-12-31,113.852785
2019,2019-12-31,102.867600
2020,2020-12-31,90.524198
2021,2021-12-31,93.622504
2022,2022-12-30,84.432786
2023,2023-12-29,109.245027
2024,2024-12-31,92.378188
2025,2025-12-31,117.675517


## Bad input
An unserved code (a financial future, a micro contract, an FMP symbol instead of a root code), a name or alias that isn't on the list (`oil` could be WTI or Brent), `quote="ALL"`, a currency with no USD pair, or a start after the end each raise a clear error.

In [15]:
week = today_et() - timedelta(days=7)
for base, quote in [("ES", "USD"), ("MGC", "USD"), ("GCUSD", "USD"), ("oil", "USD"), ("GC", "ALL"), ("GC", "XAU")]:
    try:
        commodities(base, week, quote=quote)
    except ValueError as err:
        print(f"{base}/{quote}: {err}")
try:
    commodities("GC", today_et(), week)
except ValueError as err:
    print(f"start after end: {err}")

ES/USD: ES is not a served commodity: use a code, name or alias, such as GC, gold or crude
MGC/USD: MGC is not a served commodity: use a code, name or alias, such as GC, gold or crude
GCUSD/USD: GCUSD is not a served commodity: use a code, name or alias, such as GC, gold or crude
oil/USD: OIL is not a served commodity: use a code, name or alias, such as GC, gold or crude
GC/ALL: quote must be one currency: "ALL" is for base only
GC/XAU: XAU has no stored USD pair, so USD/XAU can't be computed
start after end: start must be on or before end
